# Enhanced Combined Drought Index - Precipitation Drought Index

* **Products used:** 
[rainfall_chirps_daily](https://explorer.digitalearth.africa/products/rainfall_chirps_daily)

## Background 

Drought is an extended period, during which, fresh water availability and accessibility for the ecosystem at a given time and place is below normal, due to unfavourable spatial and temporal distribution of rainfall, temperature, soil moisture and wind characteristics [(Balint et al., 2013)](https://doi.org/10.1016/B978-0-444-59559-1.00023-2). Severe droughts can affect large populations, leading to a long-term threat to people’s livelihoods and result in tremendous economic loss [(Enenkel et al., 2016)](https://doi.org/10.3390/rs8040340).

The Enhanced Combined Drought Index aims at the timely and reliable detection of drought events with regard to their spatio-temporal extent and severity. The Enhanced Drought Index is a combination of the following: a precipitation component, which considers rainfall deficits and dryness persistence; a soil moisture component, which considers soil moisture deficits and deficit persistence; a vegetation component which considers NDVI deficits and deficit persistence; and a temperature component, which considers temperature excesses and persistence of high temperatures. The index uses satellite-derived rainfall, soil moisture, land surface temperature and vegetation status as input datasets. [(Enenkel et al., 2016)](https://doi.org/10.3390/rs8040340)

The drought index calculated using the precipitation component is referred as the **Precipitation Drought Index (PDI)**, while the index based on temperature is named as **Temperature Drought Index (TDI)**, the index based on soil moisture is named as the **Soil Moisture Drought Index** and that based on the vegetation component is named as **Vegetation Drought Index (VDI)**.

Each drought index can be expressed as:

$\text{Drought Index} = \frac{\text{Actual Average for Interest Period}}{\text{Long Term Average for Interest Period}} * \sqrt{\frac{\text{Actual Length of Continuous Deficit or Excess in the Interest Period}}{\text{Long Term Average of Continuous Deficit or Excess in the Interest Period}}}$

Each drought index is calculated similarly. The equation below illustrates the calculation of the ECDI precipitation component:

\begin{equation}
\text{PDI}_{y,d} = \frac{
\frac{1}{\text{IP}} \sum_{j=0}^{\text{IP} - 1} P^*_{y,(d-j)}}{\frac{1}{n}\sum_{k=1}^n[\frac{1}{\text{IP}} \sum_{j=0}^{\text{IP} - 1} P^*_{(d-j), k}]} * \sqrt{\frac{(\text{RL}^*)P^*_{d, y}}{\frac{1}{n}\sum_{k=1}^{n}(\text{RL}^*)P^*_{d, k}}}
\end{equation}

- $\text{PDI}_{y,d}$ is the Precipitation Drought Index for year $\text{y}$ and time unit (dekad/month) $\text{d}$

- $P^*$ is the modified dekadal/monthly precipitation average 

- $\text{RL}*$ is the modified run length parameter 

- $\text{RL*}(P*)$ (run length) is the maximum number of successive dekads/months below the long-term average rainfall in the interest period
> **Note**: For temperature, run length is the maximum number of successive dekads/months above the long-term average temperature in the interest period

- $\text{IP}$ is the interest period (e.g. 3, 4, 5, . . . dekads/months) (longer IPs detect more severe drought events). IP is flexible defines to what extent past observations are considered.

- $n$ is the number of years where relevant data are available,

- $j$ is the summation running parameter covering the Interest Period

- $k$ is the summation parameter covering the years where relevant data are available

- $d$ time unit (dekad or month) 

- $y$ year

The raw time series of temperature and precipitation as well as the run length are modified to adjust the range of all variables and to avoid a division by zero.

\begin{equation}
T^* = (T_{max} + 1) - T
\end{equation}

\begin{equation}
P^* = P + 1
\end{equation}

\begin{equation}
\text{RL}^* = (\text{RL}_{max} + 1) - \text{RL}
\end{equation}

\begin{equation}
\text{NDVI}^* = \text{NDVI} - (\text{NDVI}_{min} -0.01)
\end{equation}


- $T^*$ is the modified dekadal/monthly temperature average 
- $P^*$ is the modified dekadal/monthly precipitation average 
- $\text{RL}^*$ is the modified run length

All the individual drought indices differ in range. To improve their interpretability and visual comparability a simple scaling factor is introduced.

\begin{equation}
PDI_{scaled} = \frac{(PDI - PDI_{min})}{(PDI_{max} - PDI_{min})}
\end{equation}

            ,
The weight of each individual drought index is automatically calculated for every grid point (pixel) with respect to its capability to reflect the future vegetation status (NDVI) and multiplied by the respective individual index to calculate the ECDI. In the case of data gaps in one input dataset, the weights are automatically redistributed to other available variables.

\begin{equation}
ECDI = \sum_{i-1}^{n}w_{i} * \text{DI}_{i}
\end{equation}

- $ECDI$ Enhanced Combined Drought Index 

- $w$ Weight for each individual drought index (e.g., rainfall)

- $\text{DI}$ Individual drought index 

- $n$ number of drought indices used to calculate the ECDI 

- $i$ running parameter covering the number of drought indices

\begin{equation}
w_{i} = \frac{\frac{lag^*_{i}}{\sum_{j=1}^{n} lag^*_{j}} + \frac{corr^*_{i}}{\sum_{j=1}^{n} corr^*_{j}}}{2}
\end{equation}

- $w$ weight for the respective drought index 

- $lag^*$ modified time lag for the respective parameter 

- $corr^*$ modified correlation coefficient for the respective parameter 

- $i$ index for the respective parameter/drought index 

- $j$ running parameter covering all parameters used for the ECDI calculation

- $n$ number of individual drought indices used for the ECDI calculation


The notebook outlines:

***

## Getting started

To run this analysis, run all the cells in the notebook, starting with the "Load packages" cell. 

### Load packages

In [1]:
import calendar
import json
import os
from datetime import datetime, timedelta

import dask
import geopandas as gpd
import numpy as np
import pandas as pd
import rioxarray
import toolz
import xarray as xr
from xarray.core.groupby import DataArrayGroupBy, DatasetGroupBy
from datacube import Datacube
from deafrica_tools.bandindices import calculate_indices
from deafrica_tools.dask import create_local_dask_cluster
from deafrica_tools.datahandling import load_ard
from odc.geo.geobox import GeoBox
from odc.geo.geom import Geometry
from pyarrow import Table
from pyarrow.parquet import write_table

### Set up a Dask cluster

Dask can be used to better manage memory use and conduct the analysis in parallel. 
For an introduction to using Dask with Digital Earth Africa, see the [Dask notebook](../../Beginners_guide/08_Parallel_processing_with_dask.ipynb).

>**Note**: We recommend opening the Dask processing window to view the different computations that are being executed; to do this, see the *Dask dashboard in DE Africa* section of the [Dask notebook](../../Beginners_guide/08_Parallel_processing_with_dask.ipynb).

To use Dask, set up the local computing cluster using the cell below.

In [2]:
create_local_dask_cluster()

/usr/local/lib/python3.10/dist-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 37759 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/victoria@kartoza.com/proxy/37759/status,
Dashboard: /user/victoria@kartoza.com/proxy/37759/status,Workers: 1
Total threads: 15,Total memory: 97.21 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:38891,Workers: 1
Dashboard: /user/victoria@kartoza.com/proxy/37759/status,Total threads: 15
Started: Just now,Total memory: 97.21 GiB
Comm: tcp://127.0.0.1:36845,Total threads: 15
Dashboard: /user/victoria@kartoza.com/proxy/37053/status,Memory: 97.21 GiB
Nanny: tcp://127.0.0.1:42741,


### Analysis parameters

The following cell sets important parameters for the analysis:

* `vector_file`: file path for the shapefile containing the area of interest polygon.
* `resolution`: The x and y cell resolution of the satellite data in metres (spatial resolution). We'll use 5,000 m, which is approximately equal to the default CHIRPS resolution.
* `output_crs` : The coordinate reference system that the loaded data is to be reprojected to.
* `dask_chunks`:  the size of the dask chunks, dask breaks data into manageable chunks that can be easily stored in memory, e.g. `dict(x=1000,y=1000)`
* `time_range` : Time range to load data for.
* `ip` : The interest period to use to calculate the drought indices e.g. (3, 4, 5 dekads). It defines to what extent past observations are considered. Longer IPs detect more severe drought events. For example, if the IP=3 dekads, then the drought index (say PDI) of 0.35 for dekad 2 of 2006 implies actual drought for dekad 36 of 2005, dekad 1 of 2006 and dekad 2 of 2006.
* `output_dir` :  The directory in which to store results from the analysis.

**If running the notebook for the first time**, keep the default settings below.
This will demonstrate how the analysis works and provide meaningful results.

In [3]:
vector_file = "https://raw.githubusercontent.com/Mondieki/kenya-counties-subcounties/master/geojson/baringo.json"
resolution = (-5000, 5000)
output_crs = "EPSG:6933"
dask_chunks = dict(x=3200,y=3200)
time_range = ("2014","2024")
output_dir = "results"
# Corresponding to the six-months Standardized Precipitation Evapotranspiration Index
ip = 18

In [4]:
# Create the outpur directory if it does not exist.
os.makedirs(output_dir, exist_ok=True)

In [5]:
# Create the output path
output_path = os.path.join(output_dir, "PDI.parquet")

In [6]:
# Load the area of interest
aoi_gdf = gpd.read_file(vector_file)
geopolygon = Geometry(geom=aoi_gdf.geometry.iloc[0], crs=aoi_gdf.crs)

### Connect to the datacube

Connect to the datacube so we can access DE Africa data.
The `app` parameter is a unique name for the analysis which is based on the notebook file name.

In [7]:
# Connect to the datacube
dc = Datacube(app="DroughtIndex")

## Load CHIRPS data
Load the CHIRPS daily data from the datacube using the analysis parameters set in the previous section.

In [8]:
%%time
ds_rf = dc.load(product='rainfall_chirps_daily',
                measurements=["rainfall"],
                time=time_range,
                dask_chunks=dask_chunks,
                geopolygon=geopolygon,
                output_crs=output_crs,
                resolution=resolution,
               )
                    
ds_rf

CPU times: user 41.7 s, sys: 9.28 s, total: 51 s
Wall time: 40.9 s


<xarray.Dataset>
Dimensions:      (time: 3682, y: 49, x: 20)
Coordinates:
  * time         (time) datetime64[ns] 2014-01-01T11:59:59.500000 ... 2024-03...
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933
Data variables:
    rainfall     (time, y, x) float32 dask.array<chunksize=(1, 49, 20), meta=np.ndarray>
Attributes:
    crs:           EPSG:6933
    grid_mapping:  spatial_ref

In [9]:
# Convert to DataArray
da_rf = ds_rf["rainfall"]

# Set -9999 no-data values to NaN
da_rf = da_rf.where(da_rf !=-9999.)

da_rf

<xarray.DataArray 'rainfall' (time: 3682, y: 49, x: 20)>
dask.array<where, shape=(3682, 49, 20), dtype=float32, chunksize=(1, 49, 20), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 2014-01-01T11:59:59.500000 ... 2024-03...
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933
Attributes:
    units:         mm
    nodata:        -9999
    crs:           EPSG:6933
    grid_mapping:  spatial_ref

## Resample data

Resample the rainfall `da_rf` timeseries into **dekadal** (10-day) timesteps.

In [10]:
def get_dekad(date: np.datetime64) -> np.datetime64:
    """
    Get the start date of the dekad that a date belongs to.
    Every month has three dekads, such that the first two dekads
    have 10 days (i.e., 1-10, 11-20), and the third is comprised of the
    remaining days of the month.

    Parameters
    ----------
    date : np.datetime64
        Date to check.

    Returns
    -------
    np.datetime64
        Start date of the dekad.
    """
    timestamp = pd.Timestamp(date)
    year = timestamp.year
    month = timestamp.month

    first_day = datetime(year, month, 1)
    last_day = datetime(year, month, calendar.monthrange(year, month)[1])

    d1_start_date, d2_start_date, d3_start_date = pd.date_range(
        start=first_day, end=last_day, freq="10D", inclusive="left"
    )

    if d1_start_date <= timestamp < d2_start_date:
        return np.datetime64(d1_start_date, "ns")
    elif d2_start_date <= timestamp < d3_start_date:
        return np.datetime64(d2_start_date, "ns")
    else:
        return np.datetime64(d3_start_date, "ns")

def group_by_dekad(
    ds: xr.DataArray | xr.Dataset,
) -> DataArrayGroupBy | DatasetGroupBy:
    """
    Group a dataset or array by dekad.

    Parameters
    ----------
    ds : xr.DataArray | xr.Dataset
        Dataset or DataArray to group

    Returns
    -------
    xr.core.groupby.DataArrayGroupBy | xr.core.groupby.DatasetGroupBy
        Groupby oject
    """
    group = xr.DataArray(
        data=np.vectorize(get_dekad)(ds.time.values),
        coords=ds.time.coords,
        dims=ds.time.dims,
        name="dekad",
        attrs=ds.time.attrs,
    )
    grouped_by_dekad = ds.groupby(group)
    return grouped_by_dekad

In [11]:
%%time
# Resample the rainfall data from daily to decadal (10-day) intervals
da_rf_dekadal = group_by_dekad(da_rf).mean(dim="time").compute()
da_rf_dekadal

/usr/local/lib/python3.10/dist-packages/rasterio/warp.py:344: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  _reproject(


CPU times: user 16.8 s, sys: 504 ms, total: 17.3 s
Wall time: 1min 17s


<xarray.DataArray 'rainfall' (dekad: 363, y: 49, x: 20)>
array([[[ 0.29200163,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.29984912,  0.28720802,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.33419874,  0.31537917,  0.28235275, ...,  0.        ,
          0.        ,  0.        ],
        ...,
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.34356213,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ]],

       [[ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
...
        [ 2.5407748 ,  2.4794157 ,  2.8383396 , ...,  1.5651367 ,
          1.4777172 ,  0.8918543 ],
        [ 2.2201219 ,  2.3179328 ,  2.2798665 , ...,  1.5410626 ,
          1.9449116 ,  1.8832439 ],
        [ 2.5376573 ,  2.7804077 ,  2.3720465 , ...,  1.6634972 ,
          2.1893876 ,  2.3906493 ]],

       [[ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          2.2162352 ,  0.        ],
        ...,
        [ 9.424622  , 10.19459   , 10.981411  , ...,  0.        ,
          1.0213484 ,  0.        ],
        [ 9.818012  ,  9.534503  ,  9.844227  , ...,  1.1844184 ,
          1.1683893 ,  1.1354309 ],
        [10.010099  , 10.197859  , 10.551265  , ...,  1.6940008 ,
          1.6322978 ,  1.3778231 ]]], dtype=float32)
Coordinates:
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21
Attributes:
    units:         mm
    nodata:        -9999
    crs:           EPSG:6933
    grid_mapping:  spatial_ref

## Modify the timeseries

The raw timeseries of precipitation is modified to adjust the range of all variables to avoid a division by zero.

In [12]:
da_rf_dekadal_modified =  da_rf_dekadal + 1
da_rf_dekadal_modified

<xarray.DataArray 'rainfall' (dekad: 363, y: 49, x: 20)>
array([[[ 1.2920016,  1.       ,  1.       , ...,  1.       ,
          1.       ,  1.       ],
        [ 1.2998492,  1.2872081,  1.       , ...,  1.       ,
          1.       ,  1.       ],
        [ 1.3341987,  1.3153791,  1.2823527, ...,  1.       ,
          1.       ,  1.       ],
        ...,
        [ 1.       ,  1.       ,  1.       , ...,  1.       ,
          1.       ,  1.       ],
        [ 1.       ,  1.3435621,  1.       , ...,  1.       ,
          1.       ,  1.       ],
        [ 1.       ,  1.       ,  1.       , ...,  1.       ,
          1.       ,  1.       ]],

       [[ 1.       ,  1.       ,  1.       , ...,  1.       ,
          1.       ,  1.       ],
        [ 1.       ,  1.       ,  1.       , ...,  1.       ,
          1.       ,  1.       ],
        [ 1.       ,  1.       ,  1.       , ...,  1.       ,
          1.       ,  1.       ],
...
        [ 3.5407748,  3.4794157,  3.8383396, ...,  2.5651367,
          2.4777172,  1.8918543],
        [ 3.2201219,  3.3179328,  3.2798665, ...,  2.5410626,
          2.9449115,  2.883244 ],
        [ 3.5376573,  3.7804077,  3.3720465, ...,  2.6634972,
          3.1893876,  3.3906493]],

       [[ 1.       ,  1.       ,  1.       , ...,  1.       ,
          1.       ,  1.       ],
        [ 1.       ,  1.       ,  1.       , ...,  1.       ,
          1.       ,  1.       ],
        [ 1.       ,  1.       ,  1.       , ...,  1.       ,
          3.2162352,  1.       ],
        ...,
        [10.424622 , 11.19459  , 11.981411 , ...,  1.       ,
          2.0213485,  1.       ],
        [10.818012 , 10.534503 , 10.844227 , ...,  2.1844184,
          2.1683893,  2.1354308],
        [11.010099 , 11.197859 , 11.551265 , ...,  2.6940007,
          2.6322978,  2.377823 ]]], dtype=float32)
Coordinates:
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21

## Group the modified timeseries using the `ip` parameter

The `ip` parameter determines to what extent past observations are considered. Longer IPs detect more severe drought events. The default `ip` used of **18 dekads** corresponds to the 6-month Standardized Precipitation Evapotranspiration Index.

For example, calculating the Precipitation Drought Index for the dekad `2011-01-11` using an interest period of `ip=3` requires rainfall data from the dekad `2010-12-21`, `2011-01-01` and `2011-01-11`.


**Each interest period is labelled using its end dekad.**

In [13]:
def get_dekad_no_in_month(date: np.datetime64) -> int:
    """
    Get the number of the dekad in a month that a date belongs to.
    Every month has three dekads, such that the first two dekads
    have 10 days (i.e., 1-10, 11-20), and the third is comprised of the
    remaining days of the month.

    Parameters
    ----------
    date : np.datetime64
        Date to check.

    Returns
    -------
    int
        Number of the dekad in a month.
    """
    timestamp = pd.Timestamp(date)
    year = timestamp.year
    month = timestamp.month

    first_day = datetime(year, month, 1)
    last_day = datetime(year, month, calendar.monthrange(year, month)[1])

    d1_start_date, d2_start_date, d3_start_date = pd.date_range(
        start=first_day, end=last_day, freq="10D", inclusive="left"
    )

    if d1_start_date <= timestamp < d2_start_date:
        return 1
    elif d2_start_date <= timestamp < d3_start_date:
        return 2
    else:
        return 3


def get_dekad_no_in_year(date: np.datetime64) -> int:
    """
    Get the number of the dekad in a year that a date belongs to.
    Every month has three dekads, such that the first two dekads
    have 10 days (i.e., 1-10, 11-20), and the third is comprised of the
    remaining days of the month (21-last day). Every year has 36 dekads.

    Parameters
    ----------
    date : np.datetime64
        Date to check.

    Returns
    -------
    int
        Number of the dekad in a year.
    """
    dekad_no_in_month = get_dekad_no_in_month(date=date)
    timestamp = pd.Timestamp(date)
    month = timestamp.month
    dekad_no_in_year = ((month - 1) * 3) + dekad_no_in_month
    return dekad_no_in_year


def get_interest_period(dekad: np.datetime64, ip: int) -> np.datetime64:
    """
    Get the start and end dekad of the interest period for a dekad.
    `dekad` will always be the end dekad of the interest period.

    Parameters
    ----------
    dekad : np.datetime64
        Dekad to get the interest period for.
        Will always be the end dekad of the interest period
    ip : int
        Number of dekads in an interest period.

    Returns
    -------
    np.datetime64
        Start and end dekad of the interest period.
    """
    year = pd.Timestamp(dekad).year
    dekad_no_in_year = get_dekad_no_in_year(dekad) - (ip - 1)
    while dekad_no_in_year <= 0:
        year -= 1
        dekad_no_in_year += 36

    month = (dekad_no_in_year - 1) // 3 + 1
    dekad_no_in_month = (dekad_no_in_year - 1) % 3 + 1
    if dekad_no_in_month == 1:
        day = 1
    elif dekad_no_in_month == 2:
        day = 11
    elif dekad_no_in_month == 3:
        day = 21

    start_dekad = np.datetime64(datetime(year, month, day), "ns")
    return (start_dekad, dekad)


def bin_by_interest_period(
    ds: xr.Dataset | xr.DataArray, ip: int
) -> dict[np.datetime64, xr.Dataset | xr.DataArray]:
    """
    Bin each dekad in the dataset by interest period.

    Parameters
    ----------
    ds : xr.Dataset | xr.DataArray
        Dataset to bin
    ip : int
        Number of dekads in an interest period.

    Returns
    -------
    list[tuple[np.datetime64, np.datetime64]]
        List of dekad ranges to bin
    """
    start_date = ds.dekad.min().values
    end_date = ds.dekad.max().values
    date_range = pd.date_range(
        start_date, end_date, freq=timedelta(days=1), inclusive="both"
    ).values
    dekads = np.unique(np.vectorize(get_dekad)(date_range))
    bins = list(get_interest_period(dekad=i, ip=ip) for i in dekads)
    binned_by_interest_period = {
        end: ds.sel(dekad=slice(start, end)) for start, end in bins
    }
    return binned_by_interest_period

In [14]:
# Group the modified rainfall dekadal timeseries by the interest period.
# The dictionary keys are the end dekad for each interest period.
# The values of the dictionary are the rainfall data that belongs to the interest period
da_rf_binned_by_interest_period = bin_by_interest_period(ds=da_rf_dekadal_modified, ip=ip)

In [15]:
# Get the interest periods
interest_periods = list(da_rf_binned_by_interest_period.keys())

## Get the average values for each interest period

Get the actual average rainfall for each interest period.

In [16]:
da_rf_actual_avg_for_ip = xr.concat([da_rf_binned_by_interest_period[i].mean(dim="dekad").assign_coords(dekad=i).expand_dims({"dekad": 1}) for i in interest_periods], dim="dekad")
da_rf_actual_avg_for_ip

<xarray.DataArray 'rainfall' (dekad: 369, y: 49, x: 20)>
array([[[1.2920016, 1.       , 1.       , ..., 1.       , 1.       ,
         1.       ],
        [1.2998492, 1.2872081, 1.       , ..., 1.       , 1.       ,
         1.       ],
        [1.3341987, 1.3153791, 1.2823527, ..., 1.       , 1.       ,
         1.       ],
        ...,
        [1.       , 1.       , 1.       , ..., 1.       , 1.       ,
         1.       ],
        [1.       , 1.3435621, 1.       , ..., 1.       , 1.       ,
         1.       ],
        [1.       , 1.       , 1.       , ..., 1.       , 1.       ,
         1.       ]],

       [[1.1460009, 1.       , 1.       , ..., 1.       , 1.       ,
         1.       ],
        [1.1499245, 1.143604 , 1.       , ..., 1.       , 1.       ,
         1.       ],
        [1.1670994, 1.1576896, 1.1411763, ..., 1.       , 1.       ,
         1.       ],
...
        [4.4829135, 4.8552604, 5.346675 , ..., 4.028152 , 4.1134477,
         4.282174 ],
        [4.6004405, 5.016915 , 4.439429 , ..., 3.843149 , 4.099624 ,
         5.0744195],
        [5.1337695, 5.3067245, 5.2882113, ..., 4.2900314, 4.7853947,
         5.780925 ]],

       [[2.6244779, 2.3951905, 2.3968053, ..., 1.684229 , 1.6500363,
         1.6558833],
        [2.7968898, 2.6428645, 2.573401 , ..., 1.7260932, 1.6136231,
         1.8042831],
        [3.185643 , 3.0210404, 2.7688484, ..., 1.6584107, 1.8850683,
         1.9724536],
        ...,
        [4.883082 , 5.534899 , 6.0787687, ..., 3.7997673, 3.979492 ,
         4.070946 ],
        [5.0640593, 5.426829 , 5.0957103, ..., 3.6750257, 3.9406078,
         4.907514 ],
        [5.6229267, 5.80784  , 5.7753773, ..., 4.1103325, 4.55297  ,
         5.5532165]]], dtype=float32)
Coordinates:
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21

## Get the long term average for each interest period over the years of available data


In [17]:
# Group the interest periods by year by getting the dekad number in the year for the end dekad of
# the interest period. 
grouped_by_year = toolz.groupby(lambda dekad: get_dekad_no_in_year(date=dekad), interest_periods)

Get the long term average rainfall for each interest period.

In [18]:
long_term_avg_rf = {}
for dekad_no_in_year, interest_periods_list in grouped_by_year.items():
    long_term_avg_for_period = xr.concat([da_rf_binned_by_interest_period[i] for i in interest_periods_list], dim="dekad").mean(dim="dekad")
    long_term_avg_rf[dekad_no_in_year] = long_term_avg_for_period
    
da_rf_long_term_avg_for_ip = xr.concat([long_term_avg_rf[get_dekad_no_in_year(i)].assign_coords(dekad=i).expand_dims({"dekad": 1}) for  i in interest_periods], dim="dekad")

assert all(da_rf_actual_avg_for_ip.dekad.values == da_rf_long_term_avg_for_ip.dekad.values)

da_rf_long_term_avg_for_ip

<xarray.DataArray 'rainfall' (dekad: 369, y: 49, x: 20)>
array([[[2.7507236, 2.4500468, 2.3114686, ..., 1.3116618, 1.3749106,
         1.4813815],
        [2.9636745, 2.696412 , 2.5389245, ..., 1.3619807, 1.4258922,
         1.5707814],
        [3.2258933, 3.0636966, 2.77444  , ..., 1.4321268, 1.5358958,
         1.6278389],
        ...,
        [4.575324 , 4.641274 , 4.930497 , ..., 3.7197483, 3.7903256,
         3.9603279],
        [4.5558677, 4.706726 , 4.527445 , ..., 3.9371383, 3.7691438,
         4.130169 ],
        [4.896761 , 5.095007 , 5.0532374, ..., 3.8371177, 3.892133 ,
         4.443643 ]],

       [[2.6448786, 2.3647842, 2.2375066, ..., 1.2789761, 1.3397393,
         1.4437596],
        [2.8540728, 2.608467 , 2.4422157, ..., 1.3219744, 1.3905212,
         1.5332855],
        [3.0907307, 2.945651 , 2.6412635, ..., 1.3851899, 1.4933487,
         1.5863221],
...
        [3.1577332, 3.3235903, 3.5312552, ..., 3.0021243, 3.072863 ,
         3.2478325],
        [3.2210717, 3.2553668, 3.0311325, ..., 3.1251075, 3.2699466,
         3.539785 ],
        [3.3645403, 3.6043434, 3.547774 , ..., 3.2408574, 3.5295258,
         3.9360707]],

       [[2.125013 , 1.9563026, 1.9535156, ..., 1.5895246, 1.5515839,
         1.5692434],
        [2.2391195, 2.106691 , 2.105195 , ..., 1.6151519, 1.5337433,
         1.6512369],
        [2.4001937, 2.352779 , 2.228322 , ..., 1.547747 , 1.593801 ,
         1.7353896],
        ...,
        [3.1606774, 3.354914 , 3.5575788, ..., 3.050715 , 3.1302125,
         3.2656026],
        [3.2528756, 3.2548563, 3.0430722, ..., 3.1644168, 3.3211794,
         3.5670485],
        [3.4336734, 3.6690576, 3.6088066, ..., 3.2447107, 3.5791523,
         3.9889631]]], dtype=float32)
Coordinates:
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21

## Get the actual length of continous deficit in each interest period

For rainfall the run length is the number of successive dekads in an interest period below the long term average for the same interest period.

In [19]:
def get_no_data_mask(arr):
    """
    Check if all values in an array are NaN
    """
    return np.all(np.isnan(arr))

# From https://www.geeksforgeeks.org/maximum-consecutive-ones-or-zeros-in-a-binary-array/
def max_consecutive_ones(arr: np.ndarray) -> int:
    """
    Get the maximum number of successive ones in an array.

    Parameters
    ----------

    arr : np.ndarray
        Array to check

    Returns
    -------
    int
        Maximum number of consecutive ones in the input array.

    """

    n = len(arr)
    # initialize count
    count = 0
    # initialize max
    result = 0

    for i in range(0, n):
        # If 1 is found, increment count
        # and update result if count
        # becomes more.
        if arr[i] == 1:
            # increase count
            count += 1
            result = max(result, count)
        # Reset count if one is not found
        else:
            count = 0

    return result

In [20]:
%%time
# For each interest period get the maximum number of successive dekads below long term average rainfall in the interest period.
rf_run_length = []
for interest_period in interest_periods:
    # Get the dekads in the interest period
    ds = da_rf_binned_by_interest_period[interest_period]
    
    # Identify pixels which are empty for all dekads
    no_data_mask = xr.apply_ufunc(
        get_no_data_mask,
        ds,
        input_core_dims=[["dekad"]],
        vectorize=True,
        dask="allowed",
    )
    
    # Get the long term average for the dekads
    long_term_avg = da_rf_long_term_avg_for_ip.sel(dekad=ds.dekad)
    
    # Get the pixels where the rainfall is below the long term average rainfall
    mask = xr.where(ds < long_term_avg, 1, 0)
    
    # Get the maximum number of successive dekads below long term average rainfall.
    actual_run_length = xr.apply_ufunc(
        max_consecutive_ones,
        mask,
        input_core_dims=[["dekad"]],
        vectorize=True,
        dask="allowed",
    )
    
    # Modify the run length
    modified_run_length = (actual_run_length.max() + 1) - actual_run_length
    modified_run_length = modified_run_length.where(~no_data_mask)
    rf_run_length.append(modified_run_length.assign_coords(dekad=interest_period).expand_dims({"dekad": 1}))

da_rf_actual_run_length = xr.concat(rf_run_length, dim="dekad")
da_rf_actual_run_length

CPU times: user 6.61 s, sys: 7.86 ms, total: 6.62 s
Wall time: 6.54 s


<xarray.DataArray 'rainfall' (dekad: 369, y: 49, x: 20)>
array([[[ 1.,  1.,  1., ...,  1.,  1.,  1.],
        [ 1.,  1.,  1., ...,  1.,  1.,  1.],
        [ 1.,  1.,  1., ...,  1.,  1.,  1.],
        ...,
        [ 1.,  1.,  1., ...,  1.,  1.,  1.],
        [ 1.,  1.,  1., ...,  1.,  1.,  1.],
        [ 1.,  1.,  1., ...,  1.,  1.,  1.]],

       [[ 1.,  1.,  1., ...,  1.,  1.,  1.],
        [ 1.,  1.,  1., ...,  1.,  1.,  1.],
        [ 1.,  1.,  1., ...,  1.,  1.,  1.],
        ...,
        [ 1.,  1.,  1., ...,  1.,  1.,  1.],
        [ 1.,  1.,  1., ...,  1.,  1.,  1.],
        [ 1.,  1.,  1., ...,  1.,  1.,  1.]],

       [[ 1.,  1.,  1., ...,  1.,  1.,  1.],
        [ 1.,  1.,  1., ...,  1.,  1.,  1.],
        [ 1.,  1.,  1., ...,  1.,  1.,  1.],
        ...,
...
        ...,
        [10., 11., 11., ...,  7., 10., 10.],
        [10., 10., 10., ..., 11., 11., 10.],
        [10., 10., 10., ..., 11., 11., 12.]],

       [[ 7.,  7., 10., ...,  5.,  5.,  8.],
        [ 7.,  6.,  7., ...,  5.,  5.,  9.],
        [ 6.,  6.,  7., ...,  5.,  9.,  9.],
        ...,
        [ 9., 10., 10., ...,  6.,  9., 10.],
        [ 9.,  9.,  9., ..., 10., 10.,  9.],
        [ 9.,  9.,  9., ..., 10., 10., 11.]],

       [[ 6.,  6.,  9., ...,  5.,  5.,  8.],
        [ 6.,  5.,  6., ...,  5.,  5.,  9.],
        [ 5.,  5.,  6., ...,  5.,  9.,  9.],
        ...,
        [ 9., 10.,  9., ...,  5.,  8.,  9.],
        [ 9.,  9.,  9., ...,  9.,  9.,  8.],
        [ 9.,  9.,  8., ...,  9.,  9.,  9.]]])
Coordinates:
    spatial_ref  int32 6933
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21

## Get the long term average of continous deficit in each interest period

In [21]:
# Get the long term average rainfall run length for each interest period 
long_term_avg_run_lenth_rf = {}
for dekad_no_in_year, interest_periods_list in grouped_by_year.items():
    long_term_avg_for_period = da_rf_actual_run_length.sel(dekad=interest_periods_list).mean(dim="dekad")
    long_term_avg_run_lenth_rf[dekad_no_in_year] = long_term_avg_for_period
    
da_rf_long_term_avg_run_length_for_ip = xr.concat([long_term_avg_run_lenth_rf[get_dekad_no_in_year(i)].assign_coords(dekad=i).expand_dims({"dekad": 1}) for  i in interest_periods], dim="dekad")
da_rf_long_term_avg_run_length_for_ip

<xarray.DataArray 'rainfall' (dekad: 369, y: 49, x: 20)>
array([[[7.72727273, 6.72727273, 6.72727273, ..., 3.        ,
         4.        , 6.        ],
        [7.81818182, 7.90909091, 7.36363636, ..., 3.54545455,
         5.09090909, 6.36363636],
        [7.45454545, 8.        , 7.36363636, ..., 3.81818182,
         5.90909091, 6.45454545],
        ...,
        [6.90909091, 7.36363636, 9.27272727, ..., 8.36363636,
         8.18181818, 8.90909091],
        [6.27272727, 6.63636364, 6.72727273, ..., 8.63636364,
         8.36363636, 8.81818182],
        [6.72727273, 7.09090909, 6.72727273, ..., 8.63636364,
         8.63636364, 8.63636364]],

       [[7.45454545, 6.63636364, 6.54545455, ..., 3.45454545,
         4.36363636, 6.36363636],
        [7.36363636, 7.72727273, 7.        , ..., 3.90909091,
         5.36363636, 6.72727273],
        [7.09090909, 7.63636364, 7.27272727, ..., 4.18181818,
         6.18181818, 6.72727273],
...
        [5.63636364, 6.90909091, 7.45454545, ..., 7.        ,
         7.09090909, 8.        ],
        [5.90909091, 5.90909091, 5.90909091, ..., 7.54545455,
         7.90909091, 7.72727273],
        [6.54545455, 6.45454545, 5.81818182, ..., 7.63636364,
         8.09090909, 7.81818182]],

       [[5.18181818, 6.27272727, 6.36363636, ..., 8.72727273,
         8.45454545, 8.63636364],
        [4.72727273, 5.81818182, 5.        , ..., 8.72727273,
         8.81818182, 9.27272727],
        [4.72727273, 5.09090909, 5.72727273, ..., 8.09090909,
         9.18181818, 9.72727273],
        ...,
        [5.36363636, 6.72727273, 6.90909091, ..., 6.36363636,
         6.63636364, 7.45454545],
        [5.90909091, 5.81818182, 5.90909091, ..., 6.90909091,
         7.45454545, 7.18181818],
        [6.54545455, 6.36363636, 5.81818182, ..., 7.        ,
         7.63636364, 7.27272727]]])
Coordinates:
    spatial_ref  int32 6933
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21

## Calculate the scaled Precipitation Drought Index

In [22]:
PDI = (da_rf_actual_avg_for_ip / da_rf_long_term_avg_for_ip) * np.sqrt((da_rf_actual_run_length / da_rf_long_term_avg_run_length_for_ip))
PDI

<xarray.DataArray 'rainfall' (dekad: 369, y: 49, x: 20)>
array([[[0.16896742, 0.15736426, 0.16679863, ..., 0.44016701,
         0.36366001, 0.27558619],
        [0.15685904, 0.16974587, 0.14514567, ..., 0.38993578,
         0.31082478, 0.25236642],
        [0.15148162, 0.15179595, 0.17032797, ..., 0.35734708,
         0.26784153, 0.24179969],
        ...,
        [0.08315104, 0.07939929, 0.0666048 , ..., 0.09295845,
         0.0922356 , 0.08459646],
        [0.08763966, 0.11080867, 0.08515837, ..., 0.08642788,
         0.09174021, 0.08153473],
        [0.07873568, 0.07370624, 0.07629758, ..., 0.08868076,
         0.08742726, 0.07657647]],

       [[0.15869697, 0.16415095, 0.17468909, ..., 0.42067055,
         0.35731844, 0.27456961],
        [0.14847663, 0.15771653, 0.15476294, ..., 0.38259479,
         0.31052226, 0.25145337],
        [0.14180636, 0.14222225, 0.16021102, ..., 0.35302716,
         0.26932747, 0.24304637],
...
        [1.79393488, 1.75749705, 1.75365477, ..., 1.24223504,
         1.50811008, 1.47409574],
        [1.76262512, 1.901944  , 1.80751989, ..., 1.41572705,
         1.40974348, 1.54709727],
        [1.78921276, 1.73855644, 1.85387504, ..., 1.51480679,
         1.50731057, 1.74211923]],

       [[1.32897203, 1.19743358, 1.45909795, ..., 0.80200969,
         0.81782071, 1.01559118],
        [1.40724063, 1.16296172, 1.33907754, ..., 0.80890325,
         0.79221844, 1.07649699],
        [1.36499323, 1.27251459, 1.27181196, ..., 0.84232259,
         1.17098116, 1.09329033],
        ...,
        [2.00126878, 2.01144803, 1.95016734, ..., 1.10404731,
         1.39583384, 1.3697544 ],
        [1.92128702, 2.07368084, 2.06658558, ..., 1.32549318,
         1.30371163, 1.4520458 ],
        [1.92023686, 1.88247265, 1.87658443, ..., 1.43639272,
         1.38099645, 1.54866399]]])
Coordinates:
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21

In [23]:
PDI_scaled = (PDI - PDI.min()) / (PDI.max() - PDI.min())
PDI_scaled

<xarray.DataArray 'rainfall' (dekad: 369, y: 49, x: 20)>
array([[[0.05322236, 0.04802559, 0.05225101, ..., 0.17468581,
         0.14042026, 0.10097422],
        [0.04779932, 0.053571  , 0.04255319, ..., 0.15218851,
         0.11675669, 0.09057467],
        [0.04539091, 0.04553169, 0.05383171, ..., 0.13759285,
         0.09750558, 0.08584209],
        ...,
        [0.01478737, 0.01310705, 0.00737672, ..., 0.01917986,
         0.01885611, 0.01543473],
        [0.01679771, 0.02717452, 0.0156864 , ..., 0.01625498,
         0.01863424, 0.01406346],
        [0.01280984, 0.01055728, 0.01171787, ..., 0.01726399,
         0.01670258, 0.01184278]],

       [[0.04862248, 0.05106518, 0.05578495, ..., 0.16595384,
         0.13758003, 0.10051891],
        [0.04404505, 0.04818337, 0.04686053, ..., 0.14890067,
         0.1166212 , 0.09016573],
        [0.0410576 , 0.04124387, 0.04930059, ..., 0.13565807,
         0.09817109, 0.08640045],
...
        [0.78100428, 0.7646847 , 0.76296384, ..., 0.53391184,
         0.65299057, 0.6377564 ],
        [0.76698143, 0.82937885, 0.78708867, ..., 0.61161454,
         0.60893465, 0.67045195],
        [0.77888937, 0.75620168, 0.80784997, ..., 0.65598985,
         0.65263249, 0.75779736]],

       [[0.57275915, 0.5138464 , 0.63103927, ..., 0.33674602,
         0.34382738, 0.43240378],
        [0.60781368, 0.49840732, 0.57728516, ..., 0.33983347,
         0.33236076, 0.45968195],
        [0.58889214, 0.54747329, 0.5471586 , ..., 0.35480115,
         0.50199903, 0.46720327],
        ...,
        [0.87386391, 0.87842294, 0.85097686, ..., 0.47202105,
         0.60270487, 0.59102455],
        [0.83804209, 0.90629544, 0.90311766, ..., 0.57120106,
         0.56144566, 0.62788079],
        [0.83757175, 0.82065812, 0.81802093, ..., 0.62087017,
         0.59605958, 0.67115364]]])
Coordinates:
  * y            (y) float64 2.125e+05 2.075e+05 ... -2.25e+04 -2.75e+04
  * x            (x) float64 3.428e+06 3.432e+06 ... 3.518e+06 3.522e+06
    spatial_ref  int32 6933
  * dekad        (dekad) datetime64[ns] 2014-01-01 2014-01-11 ... 2024-03-21

In [24]:
# Convert to DataFrame.
df = PDI_scaled.to_dataframe().drop(columns="spatial_ref")
df

rainfall
dekad      y         x                  
2014-01-01  212500.0 3427500.0  0.053222
                     3432500.0  0.048026
                     3437500.0  0.052251
                     3442500.0  0.051557
                     3447500.0  0.057271
...                                  ...
2024-03-21 -27500.0  3502500.0  0.761252
                     3507500.0  0.716214
                     3512500.0  0.620870
                     3517500.0  0.596060
                     3522500.0  0.671154

[361620 rows x 1 columns]

In [25]:
# Convert the DataFrame to an Arrow table
table = Table.from_pandas(df)

# Get the tables existing metadata
existing_meta = table.schema.metadata

# Dump the crs of the DataArray as new metadata to JSON.
meta_json = json.dumps(dict(crs=str(PDI_scaled.rio.crs)))

# Combine the metadata
combined_meta = {
b"xarray.metadata": meta_json.encode(),
**existing_meta,
}

# Replace the metadata.
table = table.replace_schema_metadata(combined_meta)

In [26]:
# Write the table
write_table(table, output_path, compression="GZIP")